# 🤖 Notebook 03 — Model Training & Evaluation
Run after 01_EDA.ipynb to have cleaned_resumes.csv

In [ ]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), '..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.model_training import load_and_prepare_data, train_and_evaluate, save_model
from src.predictor      import analyze_match

print('✅ Imports OK')

In [ ]:
DATA_PATH = '../data/processed/cleaned_resumes.csv'
X_text, y, le, df = load_and_prepare_data(DATA_PATH)

In [ ]:
result = train_and_evaluate(X_text, y, le, model_type='logistic', test_size=0.20)
print(f'\n✅ Test Accuracy : {result["test_accuracy"]*100:.2f}%')
print(f'   F1 Weighted   : {result["f1_weighted"]*100:.2f}%')
print(f'   CV Mean       : {result["cv_mean"]*100:.2f}%')

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_test = result['y_test']
y_pred = result['y_pred']
cm = confusion_matrix(y_test, y_pred)
top_idx = np.argsort(np.bincount(y_test))[-10:][::-1]
top_names = le.classes_[top_idx]
cm_top = cm[np.ix_(top_idx, top_idx)]

plt.figure(figsize=(12,8))
sns.heatmap(cm_top, annot=True, fmt='d', cmap='Blues',
            xticklabels=[n[:12] for n in top_names],
            yticklabels=[n[:12] for n in top_names],
            linewidths=0.5, linecolor='white')
plt.title(f'Confusion Matrix (Top 10) — Accuracy: {result["test_accuracy"]*100:.1f}%')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../screenshots/plot7_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
report = result['report_dict']
classes = [k for k in report if k not in ['accuracy','macro avg','weighted avg'] and isinstance(report[k],dict)]
f1s = [report[c]['f1-score'] for c in classes]
sorted_idx = np.argsort(f1s)
classes_s = [classes[i] for i in sorted_idx]
f1s_s     = [f1s[i]     for i in sorted_idx]
colors    = ['#E74C3C' if f<0.7 else '#F39C12' if f<0.85 else '#27AE60' for f in f1s_s]
plt.figure(figsize=(12,7))
plt.barh(range(len(classes_s)), f1s_s, color=colors, edgecolor='white')
plt.yticks(range(len(classes_s)), [c[:20] for c in classes_s], fontsize=8)
plt.xlabel('F1 Score')
plt.title('Per-Class F1 Score')
plt.axvline(0.85, color='green', linestyle='--', alpha=0.5, label='85% threshold')
plt.xlim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.savefig('../screenshots/plot8_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
metadata = {
    'model_type'      : result['model_type'],
    'test_accuracy'   : f"{result['test_accuracy']*100:.2f}%",
    'f1_weighted'     : f"{result['f1_weighted']*100:.2f}%",
    'cv_mean'         : f"{result['cv_mean']*100:.2f}%",
    'n_classes'       : len(le.classes_),
    'classes'         : list(le.classes_),
    'n_train_samples' : len(result['X_train']),
    'n_test_samples'  : len(result['X_test']),
    'train_time'      : result['train_time'],
}
save_model(result['pipeline'], le, metadata)
print('\n✅ Model saved to model/ folder!')

In [ ]:
test_resume = '''
Senior Data Scientist 4 years experience.
Python, SQL, Machine Learning, Deep Learning,
TensorFlow, NLP, Docker, AWS, Git, Pandas, Flask
B.Tech Computer Science 2020
'''
test_jd = '''
Data Scientist needed 3+ years.
Must: Python, SQL, Machine Learning, TensorFlow, NLP, Docker, AWS.
'''
r = analyze_match(test_resume, test_jd)
print(f'Score    : {r["final_score_pct"]}%  {r["score_emoji"]}')
print(f'Category : {r["predicted_category"]} ({r["confidence"]}%)')
print(f'Matched  : {r["matched_skills"][:5]}')
print(f'Missing  : {r["missing_skills"][:5]}')
print('\n✅ End-to-end test passed! Ready for Flask app (Stage 5)')